In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [26]:
df = pd.read_csv("static/data/GDP.csv", skiprows=4)
df.head()

european_countries = ["ALB", "AND", "AUT", "BLR", "BEL", "BIH", "BGR", "HRV", "CYP", "CZE", "DNK", "EST", "FRO", 
                      "FIN", "FRA", "DEU", "GIB", "GRC", "HUN", "ISL", "IRL", "IMN", "ITA", "XKX", "LVA", "LIE", 
                      "LTU", "LUX", "MKD", "MLT", "MDA", "MCO", "MNE", "NLD", "NOR", "POL", "PRT", "ROU", "RUS", 
                      "SMR", "SRB", "SVK", "SVN", "ESP", "SWE", "CHE", "UKR", "GBR", "VAT", "RSB"]
df = df[df["Country Code"].isin(european_countries)]
df_expenditure = pd.read_csv("static/data/expenditure.csv")
df_completion_rate_pr_ed = pd.read_csv("static/data/Completion_Rate_Primary_Ed.csv")
df_completion_rate_low_sec_ed = pd.read_csv("static/data/Completion_Rate_Lower_Secondary_Ed.csv")
df_completion_rate_high_sec_ed = pd.read_csv("static/data/Completion_Rate_Upper_Secondary_Ed.csv")

In [27]:
df = df.drop(columns=['Indicator Code', 'Indicator Name', 'Unnamed: 70']).iloc[:,0:69]
df = df.melt(id_vars=['Country Code', 'Country Name'], var_name='year', value_name='GDP (current US$)').astype({'year': int})
df = df[df['year']>=2000].reset_index(drop=True)
print(df.head())

  Country Code Country Name  year  GDP (current US$)
0          ALB      Albania  2000       3.584570e+09
1          AND      Andorra  2000       1.432606e+09
2          AUT      Austria  2000       1.961816e+11
3          BEL      Belgium  2000       2.367925e+11
4          BGR     Bulgaria  2000       1.324599e+10


In [28]:
from scipy.stats import linregress
def impute_with_linear_fit(df):
    df_imputed = df.copy()
    years = np.array(df_imputed.columns.astype(int))
    for country in df_imputed.index:
        row = df.loc[country]
        valid = row.notna()
        valid_years = years[valid]
        valid_values = row[valid]
        if len(valid_values) < 2:
            continue
        slope, intercept,_,_,_ = linregress(valid_years, valid_values)

        nan = row.isna()
        nan_years = years[nan]
        df_imputed.loc[country, nan] = nan_years * slope + intercept
    return df_imputed

In [29]:
df_e = df_expenditure.rename(columns={'value': 'Expenditure', 'geoUnit':'Country Code'}).iloc[:,1:4]
df_e = df_e.pivot_table(index='Country Code', columns='year', values='Expenditure')
df_e = impute_with_linear_fit(df_e)
scaler = MinMaxScaler()
df_e.iloc[:,:] = scaler.fit_transform(df_e)
df_e = df_e.reset_index()
df_e = df_e.melt(id_vars=['Country Code'], var_name='year', value_name='Expenditure').astype({'year': int})
print(df_e.head())

  Country Code  year  Expenditure
0          ALB  2000     0.646280
1          AND  2000     0.507259
2          AUT  2000     0.703416
3          BEL  2000     0.743234
4          BGR  2000     0.692841


In [30]:
df_completion_rate_pr_ed =df_completion_rate_pr_ed.rename(columns={'value': 'Completion Rate: Primary Education', 'geoUnit':'Country Code'}).iloc[:,1:4]
df_completion_rate_pr_ed = df_completion_rate_pr_ed.pivot_table(index='Country Code', columns='year', values='Completion Rate: Primary Education')
df_completion_rate_pr_ed = impute_with_linear_fit(df_completion_rate_pr_ed)
scaler = MinMaxScaler()
df_completion_rate_pr_ed.iloc[:,:] = scaler.fit_transform(df_completion_rate_pr_ed)
df_primary_ed = df_completion_rate_pr_ed.reset_index().melt(id_vars=['Country Code'], var_name='year', value_name='Completion Rate: Primary Education').astype({'year': int})
df_primary_ed.head()

,Country Code,year,Completion Rate: Primary Education
0,ALB,2000,0.621562
1,AUT,2000,0.940323
2,BEL,2000,NaN
3,BGR,2000,0.783052
4,BIH,2000,0.912182


In [31]:
df_completion_rate_lower_sec_ed =df_completion_rate_low_sec_ed.rename(columns={'value': 'Completion Rate: Lower Secondary Education', 'geoUnit':'Country Code'}).iloc[:,1:4]
df_completion_rate_lower_sec_ed = df_completion_rate_lower_sec_ed.pivot_table(index='Country Code', columns='year', values='Completion Rate: Lower Secondary Education')
df_completion_rate_lower_sec_ed = impute_with_linear_fit(df_completion_rate_lower_sec_ed)
scaler = MinMaxScaler()
df_completion_rate_lower_sec_ed.iloc[:,:] = scaler.fit_transform(df_completion_rate_lower_sec_ed)
df_lower_sec = df_completion_rate_lower_sec_ed.reset_index().melt(id_vars=['Country Code'], var_name='year', value_name='Completion Rate: Lower Secondary Education').astype({'year': int})
df_lower_sec.head()

,Country Code,year,Completion Rate: Lower Secondary Education
0,ALB,2000,0.693348
1,AUT,2000,0.936748
2,BEL,2000,0.085523
3,BGR,2000,0.751377
4,BIH,2000,0.868227


In [32]:
df_completion_rate_higher_sec_ed =df_completion_rate_high_sec_ed.rename(columns={'value': 'Completion Rate: Higher Secondary Education', 'geoUnit':'Country Code'}).iloc[:,1:4]
df_completion_rate_higher_sec_ed = df_completion_rate_higher_sec_ed.pivot_table(index='Country Code', columns='year', values='Completion Rate: Higher Secondary Education')
df_completion_rate_higher_sec_ed = impute_with_linear_fit(df_completion_rate_higher_sec_ed)
scaler = MinMaxScaler()
df_completion_rate_higher_sec_ed.iloc[:,:] = scaler.fit_transform(df_completion_rate_higher_sec_ed)
df_higher_sec = df_completion_rate_higher_sec_ed.reset_index().melt(id_vars=['Country Code'], var_name='year', value_name='Completion Rate: Higher Secondary Education').astype({'year': int})
df_higher_sec.head()

,Country Code,year,Completion Rate: Higher Secondary Education
0,ALB,2000,0.222819
1,AUT,2000,0.832271
2,BEL,2000,0.762887
3,BGR,2000,0.844335
4,BIH,2000,0.313493


In [36]:
df_merged = pd.merge(df, df_e, on=['Country Code', 'year'], how='outer')
df_merged = pd.merge(df_merged, df_primary_ed, on=['Country Code', 'year'], how='outer')
df_merged = pd.merge(df_merged, df_lower_sec, on=['Country Code', 'year'], how='outer')
df_merged = pd.merge(df_merged, df_higher_sec, on=['Country Code', 'year'], how='outer')
print(df_merged.head())

  Country Code Country Name  year  GDP (current US$)  Expenditure  \
0          ALB      Albania  2000       3.584570e+09     0.646280   
1          ALB      Albania  2001       4.059064e+09     0.515367   
2          ALB      Albania  2002       4.515003e+09     0.400440   
3          ALB      Albania  2003       5.801712e+09     0.496162   
4          ALB      Albania  2004       7.406646e+09     0.398810   

   Completion Rate: Primary Education  \
0                            0.621562   
1                                 NaN   
2                                 NaN   
3                                 NaN   
4                                 NaN   

   Completion Rate: Lower Secondary Education  \
0                                    0.693348   
1                                         NaN   
2                                         NaN   
3                                         NaN   
4                                         NaN   

   Completion Rate: Higher Secondary Educat